In [2]:
cd ..

/home/karaman/Documents/bet


In [3]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import src.utils.config as config



In [4]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/25 00:00:16 WARN Utils: Your hostname, karaman-VMware-Virtual-Platform, resolves to a loopback address: 127.0.1.1; using 192.168.182.129 instead (on interface ens33)
26/01/25 00:00:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/25 00:00:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/25 00:00:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StringIndexer, OneHotEncoder

In [7]:
model_df = (
    player_behavior
        .join(
            player_snapshot,
            on="player_idx",
            how="left"
        )
        .join(
            labels,
            on=["player_idx", "reference_date"],
            how="inner"   # label must exist
        )
)
player_snapshot.columns

['player_idx',
 'country',
 'age_bucket',
 'device_type',
 'acquisition_channel',
 'registration_date',
 'risk_segment',
 'lifecycle_stage',
 'current_balance',
 'first_event_date',
 'last_event_date',
 'first_session_date',
 'last_session_date',
 'first_financial_date',
 'last_financial_date']

In [ ]:
categorical_cols = [
    "country",
    "age_bucket",
    "device_type",
    "acquisition_channel",
    "risk_segment",
    "lifecycle_stage"
]
